# Machine Learning for Climate Data
## Future Tech Leaders — Syria | Lesson 1

### Learning objectives
By the end of this practical session, you will be able to:

1. Explain the difference between AI, ML, and deep learning.
2. Identify features, targets, observations, and datasets.
3. Distinguish regression, classification, and clustering.
4. Train a machine learning model using climate-related data.
5. Evaluate predictions on unseen data.
6. Explain overfitting, generalisation, and data leakage.

### Our climate problem
Can we predict daily solar energy generation using weather data?

We will build a simple ML pipeline, evaluate its performance, and discuss how the same principles apply to real climate data.

**Important:** The dataset is synthetic. It demonstrates ML concepts but is not suitable for operational energy or climate decisions.

## 1. Import libraries

These are common Python tools for data analysis, visualisation, and machine learning.

In [ ]:
# NumPy provides numerical operations and random number generation
import numpy as np

# Pandas stores and analyses data in table-like DataFrames
import pandas as pd

# Matplotlib creates charts
import matplotlib.pyplot as plt

# Split data into training and testing periods
from sklearn.model_selection import train_test_split

# Metrics measure how close predictions are to the true values
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Supervised learning algorithms
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Unsupervised learning algorithm for finding groups
from sklearn.cluster import KMeans

# Set a random seed so the synthetic data can be reproduced
np.random.seed(42)

# Display plots inside the notebook
%matplotlib inline

print("Libraries imported successfully.")

## 2. AI, ML, and deep learning

- **Artificial Intelligence (AI):** the broad field of intelligent computational systems.
- **Machine Learning (ML):** methods that learn patterns from data.
- **Deep Learning (DL):** ML methods that use neural networks with multiple layers.

### Three learning approaches
- **Supervised learning:** learns from inputs paired with known targets.
- **Unsupervised learning:** discovers patterns without task-specific target labels.
- **Reinforcement learning:** learns actions through interaction and rewards.

### Our task
We have historical weather observations and measured solar energy generation.

- Task: predict a numerical quantity.
- Learning approach: supervised learning.
- ML task: regression.
- Features: weather measurements and installed solar capacity.
- Target: daily solar energy generation in kWh.

## 3. Create a synthetic climate dataset

We generate two years of daily observations so the notebook runs without downloading a file.

In [ ]:
# Create 730 consecutive daily dates (approximately two years)
dates = pd.date_range(start="2023-01-01", periods=730, freq="D")

# Day number helps us simulate seasonal changes through the year
day_of_year = dates.dayofyear.to_numpy()

# Simulate seasonal temperature changes plus random day-to-day variation
temperature = (
    18
    + 10 * np.sin(2 * np.pi * (day_of_year - 100) / 365)
    + np.random.normal(0, 3, len(dates))  # Random noise makes observations less perfectly regular
)

# Simulate a seasonal pattern in solar radiation
seasonal_radiation = 4.5 + 2.0 * np.sin(
    2 * np.pi * (day_of_year - 80) / 365
)

# Generate a cloud-cover value between 0% and 100% for each day
cloud_cover = np.random.uniform(0, 100, len(dates))

# Higher cloud cover generally reduces the simulated solar radiation
solar_radiation = (
    seasonal_radiation * (1 - 0.006 * cloud_cover)
    + np.random.normal(0, 0.4, len(dates))
)

# Clip values to keep simulated radiation within the chosen range
solar_radiation = np.clip(solar_radiation, 0.1, 8.0)

# Simulate humidity and rainfall; these are illustrative, not a weather model
humidity = np.clip(
    65 - 2 * (temperature - 18) + np.random.normal(0, 10, len(dates)),
    15, 100  # Keep humidity between 15% and 100%
)

rainfall = np.maximum(
    0,
    np.random.gamma(shape=1.2, scale=2.0, size=len(dates)) * (humidity / 70)
)

# Assume a solar installation with a constant capacity of 10 kW
installed_capacity_kw = 10

# Generate energy output using a simplified relationship with radiation and temperature
solar_generation = (
    solar_radiation
    * installed_capacity_kw
    * 0.75
    * (1 - 0.002 * np.maximum(temperature - 25, 0))
    + np.random.normal(0, 1.5, len(dates))
)

# Energy generation cannot be negative, so replace negative values with zero
solar_generation = np.clip(solar_generation, 0, None)

# Combine all simulated variables into one table; each row represents one day
df = pd.DataFrame({
    "date": dates,
    "temperature_c": temperature,
    "solar_radiation_kwh_m2": solar_radiation,
    "cloud_cover_percent": cloud_cover,
    "humidity_percent": humidity,
    "rainfall_mm": rainfall,
    "installed_capacity_kw": installed_capacity_kw,
    "solar_generation_kwh": solar_generation
})

# Show the first five rows to inspect the table
df.head()

## 4. Understand the dataset

Each row is an observation; each column is a variable. Before modelling, inspect data types, summary statistics, and missing values.

In [ ]:
# shape returns (number of rows, number of columns)
print("Dataset shape:", df.shape)

# info() shows column names, data types, and how many values are present
df.info()

# describe() calculates summaries such as mean, standard deviation, and quartiles
display(df.describe().round(2))

# isnull().sum() counts missing values in each column
print("\nMissing values:")
print(df.isnull().sum())

# duplicated().sum() counts rows that are exact duplicates
print("\nNumber of duplicate rows:", df.duplicated().sum())

## 5. Visualise climate variables

Plots help us inspect seasonal patterns and relationships before training a model.

In [ ]:
# Create two vertically stacked plots that share the same date axis
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Plot the first 90 daily temperature values
axes[0].plot(df["date"][:90], df["temperature_c"][:90])
axes[0].set_title("Daily Temperature")
axes[0].set_ylabel("Temperature (°C)")
axes[0].grid(True)

# Plot solar radiation for the same 90 days
axes[1].plot(df["date"][:90], df["solar_radiation_kwh_m2"][:90])
axes[1].set_title("Daily Solar Radiation")
axes[1].set_ylabel("Radiation (kWh/m²)")
axes[1].set_xlabel("Date")
axes[1].grid(True)

plt.tight_layout()  # Reduce overlap between plot labels
plt.show()

# A scatter plot shows the relationship between two numerical variables
plt.figure(figsize=(8, 5))
plt.scatter(
    df["solar_radiation_kwh_m2"],
    df["solar_generation_kwh"],
    alpha=0.6  # Transparency helps when points overlap
)
plt.xlabel("Solar radiation (kWh/m²)")
plt.ylabel("Energy generated (kWh)")
plt.title("Solar Radiation vs Solar Energy Generation")
plt.grid(True)
plt.show()

## 6. Features, target, and notation

- **Observation:** one row, representing one day.
- **Feature:** an input variable used by the model.
- **Target:** the variable we want the model to predict.
- **Training:** fitting a model to examples.
- **Inference:** using a fitted model to predict for new inputs.
- **Parameter:** a value learned during fitting.
- **Hyperparameter:** a setting chosen before/during model selection.

We use **X** for the feature matrix and **y** for the target vector. If there are *m* observations and *n* features, X has shape `(m, n)`.

In [ ]:
# List the input variables the model is allowed to use
features = [
    "temperature_c",
    "solar_radiation_kwh_m2",
    "cloud_cover_percent",
    "humidity_percent",
    "installed_capacity_kw"
]

# The target is what we want the model to predict
target = "solar_generation_kwh"

X = df[features]  # Uppercase X commonly denotes a feature matrix
y = df[target]    # Lowercase y commonly denotes the target vector

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

# Use the first 80% of dates for training and the last 20% for testing
# A chronological split is often more realistic for forecasting than a random split
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]  # Earlier input rows
X_test = X.iloc[split_index:]   # Later input rows held back for evaluation

y_train = y.iloc[:split_index]  # Known targets for training
y_test = y.iloc[split_index:]   # Targets used only to evaluate predictions

print("\nTraining observations:", len(X_train))
print("Testing observations:", len(X_test))

print("\nTraining period:")
print(df["date"].iloc[0], "to", df["date"].iloc[split_index - 1])

print("\nTesting period:")
print(df["date"].iloc[split_index], "to", df["date"].iloc[-1])

## 7. Regression: predict a numerical value

Regression predicts a continuous number, such as energy generation, temperature, or rainfall.

A simple linear model can be written as:

`ŷ = w₁x₁ + w₂x₂ + ... + wₙxₙ + b`

Here, `x` values are features, `w` values are learned coefficients, `b` is the intercept, and `ŷ` is the prediction.

In [ ]:
# Baseline: predict the average energy generation seen during training for every test day
# A baseline gives us a simple reference to beat
baseline_prediction = np.full(
    shape=len(y_test),  # Make one prediction for each test observation
    fill_value=y_train.mean()
)

# Create a linear regression model
linear_model = LinearRegression()

# fit() learns the model coefficients from the training features and known targets
linear_model.fit(X_train, y_train)

# predict() applies the fitted model to inputs it did not train on
linear_predictions = linear_model.predict(X_test)

print("Model trained successfully.")

# coef_ contains the learned coefficient for each feature
coefficients = pd.DataFrame({
    "Feature": features,
    "Learned coefficient": linear_model.coef_
})

display(coefficients.round(3))

# intercept_ is the model's learned constant term
print("Intercept:", round(linear_model.intercept_, 3))

## 8. Evaluate regression

- **MAE:** average absolute error, in kWh.
- **RMSE:** square root of average squared error; large errors count more heavily.
- **R²:** compares the model against predicting the mean target. Test R² can be negative.

Lower MAE and RMSE are generally better. Interpret all metrics in the context of the intended decision.

In [ ]:
# Create a reusable function so we can evaluate several models consistently
def evaluate_regression(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)  # Average absolute difference

    # MSE squares each error; taking the square root returns the metric to kWh units
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # R² compares prediction quality with a mean-based reference
    r2 = r2_score(y_true, y_pred)

    return {
        "Model": model_name,
        "MAE (kWh)": mae,
        "RMSE (kWh)": rmse,
        "R²": r2
    }

# Compare the simple baseline with the trained linear regression model
results = pd.DataFrame([
    evaluate_regression(y_test, baseline_prediction, "Mean baseline"),
    evaluate_regression(y_test, linear_predictions, "Linear regression")
])

display(results.round(3))

In [ ]:
# Put observed values and predictions side by side for inspection
comparison = pd.DataFrame({
    "Date": df["date"].iloc[split_index:].to_numpy(),
    "Actual energy (kWh)": y_test.to_numpy(),
    "Predicted energy (kWh)": linear_predictions
})

display(comparison.head(10).round(2))

# Plot actual and predicted values across the test period
plt.figure(figsize=(12, 5))
plt.plot(comparison["Date"], comparison["Actual energy (kWh)"], label="Actual", alpha=0.8)
plt.plot(comparison["Date"], comparison["Predicted energy (kWh)"], label="Predicted", alpha=0.8)
plt.title("Actual vs Predicted Solar Energy Generation")
plt.xlabel("Date")
plt.ylabel("Energy generation (kWh)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 9. Decision trees and overfitting

A decision tree learns a sequence of feature-based splits. `max_depth` is a hyperparameter that limits how many levels the tree can grow.

A very deep tree may fit training data extremely well but perform poorly on unseen data. This is **overfitting**. **Generalisation** means performing well on relevant new examples.

In [ ]:
# Define models with different complexity levels
models = {
    "Linear regression": LinearRegression(),
    "Decision tree (depth 2)": DecisionTreeRegressor(max_depth=2, random_state=42),
    "Decision tree (depth 5)": DecisionTreeRegressor(max_depth=5, random_state=42),
    "Decision tree (unrestricted)": DecisionTreeRegressor(random_state=42)
}

model_results = []

for name, model in models.items():
    # Fit each model using only the training period
    model.fit(X_train, y_train)

    # Training error tells us how well the model fits the data it learned from
    train_predictions = model.predict(X_train)
    train_mae = mean_absolute_error(y_train, train_predictions)

    # Test error checks performance on later observations not used for fitting
    test_predictions = model.predict(X_test)
    test_mae = mean_absolute_error(y_test, test_predictions)

    model_results.append({
        "Model": name,
        "Training MAE": train_mae,
        "Testing MAE": test_mae
    })

model_comparison = pd.DataFrame(model_results)
display(model_comparison.round(3))

# Plot training and testing errors to help identify possible overfitting
model_comparison.plot(
    x="Model",
    y=["Training MAE", "Testing MAE"],
    kind="bar",
    figsize=(11, 5)
)
plt.ylabel("Mean absolute error (kWh)")
plt.title("Training Error vs Testing Error")
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

### Discussion
If a deep tree has very low training error but substantially higher test error, what does that tell you? Which model would you choose, and why?

A more complex model is not automatically better. In a full ML workflow, use validation data to select model settings, and keep the test set for final evaluation.

## 10. Classification: predict a category

Classification predicts categories rather than continuous values.

For this demonstration, a day is labelled **high-rainfall** if rainfall exceeds 5 mm. This is a teaching threshold, not an official meteorological definition.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, ConfusionMatrixDisplay
)

# Create a binary target: 1 when rainfall is above 5 mm, otherwise 0
df["high_rainfall"] = (df["rainfall_mm"] > 5).astype(int)

# Do not include rainfall_mm as an input because it directly defines the target
classification_features = [
    "temperature_c",
    "solar_radiation_kwh_m2",
    "cloud_cover_percent",
    "humidity_percent"
]

X_class = df[classification_features]
y_class = df["high_rainfall"]

# Keep the same chronological training/test boundary
Xc_train = X_class.iloc[:split_index]
Xc_test = X_class.iloc[split_index:]
yc_train = y_class.iloc[:split_index]
yc_test = y_class.iloc[split_index:]

# Pipeline applies scaling and then classification in sequence
# StandardScaler learns feature means/scales from training data only
classifier = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

classifier.fit(Xc_train, yc_train)  # Learn the relationship between weather and class
class_predictions = classifier.predict(Xc_test)  # Predict classes for later dates

# Accuracy is the fraction of predictions that are correct
print("Accuracy:", round(accuracy_score(yc_test, class_predictions), 3))

# Precision measures how often predicted positive cases are truly positive
print("Precision:", round(
    precision_score(yc_test, class_predictions, zero_division=0), 3
))

# Recall measures how many actual positive cases the model detects
print("Recall:", round(
    recall_score(yc_test, class_predictions, zero_division=0), 3
))

# F1 balances precision and recall in a single score
print("F1 score:", round(
    f1_score(yc_test, class_predictions, zero_division=0), 3
))

# The confusion matrix shows correct and incorrect predictions by class
ConfusionMatrixDisplay.from_predictions(
    yc_test,
    class_predictions,
    display_labels=["Not high-rainfall", "High-rainfall"]
)
plt.title("High-Rainfall Classification")
plt.tight_layout()
plt.show()

### Discussion
In an early-warning application, why might recall be important? What is the consequence of incorrectly classifying a high-rainfall day as a normal day?

This synthetic model is for learning only; it is not a reliable rainfall-warning system.

## 11. Unsupervised learning: clustering

Clustering looks for groups in data without supplying target labels. K-means assigns observations to a chosen number of groups based on distances to cluster centres.

Feature scaling matters because variables with larger numerical ranges can otherwise dominate distance calculations.

In [ ]:
# Select climate variables; no target labels are supplied to K-means
climate_features = [
    "temperature_c",
    "solar_radiation_kwh_m2",
    "humidity_percent",
    "rainfall_mm"
]
X_climate = df[climate_features]

# Standardise each variable so different units/scales are more comparable
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_climate)

# Ask K-means to identify three clusters; three is a teaching choice
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

# fit_predict() learns the clusters and assigns a cluster ID to each row
df["climate_cluster"] = kmeans.fit_predict(X_scaled)

# Compare average climate measurements in each discovered cluster
cluster_summary = df.groupby("climate_cluster")[climate_features].mean()
display(cluster_summary.round(2))

# Display the clusters using only temperature and rainfall as a 2D view
plt.figure(figsize=(9, 6))
for cluster_id in sorted(df["climate_cluster"].unique()):
    subset = df[df["climate_cluster"] == cluster_id]
    plt.scatter(
        subset["temperature_c"],
        subset["rainfall_mm"],
        label=f"Cluster {cluster_id}",
        alpha=0.6
    )

plt.xlabel("Temperature (°C)")
plt.ylabel("Rainfall (mm)")
plt.title("K-means: Climate Observation Clusters")
plt.legend()
plt.grid(True)
plt.show()

### Reflection
Are the clusters clearly separated? Do the groups correspond to meaningful weather patterns? What additional data or domain knowledge would help you interpret them?

The algorithm finds groups according to its mathematical objective. Domain knowledge is needed to decide whether the groups are useful.

## 12. From a model to a climate product

Before selecting an algorithm, define the problem carefully.

Example: forecasting next-day solar energy generation.

1. User: solar-system operator.
2. Decision: plan battery charging.
3. Observation: one solar site and one forecast-origin day.
4. Features: information genuinely available at forecast time.
5. Target: energy generation on the following day.
6. Task: supervised regression.
7. Baseline: a simple historical or persistence forecast.
8. Evaluation: MAE and RMSE on later dates.

### Important limitations
- Synthetic data are not a substitute for measured climate observations.
- Historical correlation does not prove causation.
- Random train-test splitting can be misleading for time-dependent data.
- Data leakage occurs when information unavailable at prediction time enters training or evaluation.
- A model should be compared with a sensible baseline.
- Deployment requires data-quality checks, validation, monitoring, and an assessment of the consequences of errors.

## 13. Student exercise

Change the features below and investigate how model performance changes.

1. Choose different features.
2. Train a regression model on the training period.
3. Evaluate it on the test period using MAE and RMSE.
4. Compare it with the mean baseline.
5. Explain whether your model is useful and what limitations remain.

In [ ]:
# Change this list to experiment with different input variables
student_features = [
    "solar_radiation_kwh_m2",
    "cloud_cover_percent"
]

X_student = df[student_features]
y_student = df["solar_generation_kwh"]

# Reuse the same chronological split to make the comparison fair
X_student_train = X_student.iloc[:split_index]
X_student_test = X_student.iloc[split_index:]
y_student_train = y_student.iloc[:split_index]
y_student_test = y_student.iloc[split_index:]

# Train a new model using only the selected features
student_model = LinearRegression()
student_model.fit(X_student_train, y_student_train)

# Generate predictions for the held-out test period
student_predictions = student_model.predict(X_student_test)

# Baseline predicts the average target value from the training period
student_baseline = np.full(
    len(y_student_test),
    y_student_train.mean()
)

# Compare both approaches using the same evaluation function
display(pd.DataFrame([
    evaluate_regression(y_student_test, student_baseline, "Mean baseline"),
    evaluate_regression(y_student_test, student_predictions, "Student model")
]).round(3))

# Lesson recap

| Concept | Meaning | Climate example |
|---|---|---|
| Feature | Input variable | Solar radiation |
| Target | Variable to predict | Energy generated |
| Supervised learning | Learns from labelled examples | Energy forecasting |
| Regression | Predicts a numerical value | Temperature prediction |
| Classification | Predicts a category | Rainfall category |
| Clustering | Finds groups without target labels | Climate profiles |
| Training set | Data used to fit a model | Earlier observations |
| Test set | Data used for final evaluation | Later observations |
| MAE / RMSE | Regression error metrics | Error in kWh |
| Overfitting | Fits training data but generalises poorly | Deep tree with high test error |
| Data leakage | Uses information that should not be available | Including future observations |
| Generalisation | Performance on relevant unseen data | Forecasting future days |

## Final question

If you were building an AI system to help a Syrian community manage water resources during a drought:

1. What would you predict?
2. Which features would you collect?
3. Would you use regression, classification, or clustering?
4. What baseline would you compare against?
5. How would you measure whether the model helps the community?